# Multiresolution hash grid — the Instant-NGP trick

**Track B · 3D & Neural Rendering** · maps to lesson B6 (hash grids).

Instant-NGP made NeRF ~1000× faster by replacing a big MLP with a **multiresolution hash grid** of trainable features + a tiny MLP. Here we fit a 2D image with it (the paper's gigapixel demo) so it trains in seconds.

> CPU is fine; PSNR climbs fast.

In [ ]:
import os, math, torch, torch.nn as nn, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 1500)); H = W = 96
L, T, Fdim = 8, 1 << 14, 2                      # levels, hash-table size, features/level
res = [int(round(8 * 1.5 ** l)) for l in range(L)]
primes = torch.tensor([1, 2654435761], device=device)

## 1 · Target image (procedural, self-contained)

In [ ]:
ys = torch.linspace(0, 1, H, device=device); xs = torch.linspace(0, 1, W, device=device)
YY, XX = torch.meshgrid(ys, xs, indexing="ij")
target = torch.stack([0.5 + 0.5 * torch.sin(12 * XX) * torch.cos(9 * YY),
                      0.5 + 0.5 * torch.sin(7 * (XX + YY)),
                      0.5 + 0.5 * torch.cos(10 * YY)], -1).clamp(0, 1)
coords = torch.stack([XX, YY], -1).reshape(-1, 2)
plt.imshow(target.cpu()); plt.title("target"); plt.axis("off"); plt.show()

## 2 · The hash-grid encoding (bilinear, per level)

In [ ]:
tables = nn.Parameter((torch.rand(L, T, Fdim, device=device) * 2 - 1) * 1e-4)
def encode(c):                                  # c (N,2) in [0,1]
    out = []
    for l in range(L):
        p = c * (res[l] - 1)
        x0 = p[:, 0].floor().long(); y0 = p[:, 1].floor().long(); x1 = x0 + 1; y1 = y0 + 1
        wx = (p[:, 0] - x0).unsqueeze(1); wy = (p[:, 1] - y0).unsqueeze(1)
        def f(ix, iy): return tables[l][((ix * primes[0]) ^ (iy * primes[1])) % T]
        c0 = f(x0, y0) * (1 - wx) + f(x1, y0) * wx
        c1 = f(x0, y1) * (1 - wx) + f(x1, y1) * wx
        out.append(c0 * (1 - wy) + c1 * wy)
    return torch.cat(out, -1)
mlp = nn.Sequential(nn.Linear(L * Fdim, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 3)).to(device)

## 3 · Train — fit the image

In [ ]:
opt = torch.optim.Adam([tables, *mlp.parameters()], 1e-2); tgt = target.reshape(-1, 3); hist = []
for step in range(STEPS + 1):
    rgb = torch.sigmoid(mlp(encode(coords)))
    loss = ((rgb - tgt) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0:
        psnr = (-10 * torch.log10(loss)).item(); hist.append((step, psnr)); print(f"step {step:4d}  PSNR {psnr:5.2f} dB")

## 4 · Compare — reconstruction vs. target

In [ ]:
with torch.no_grad(): img = torch.sigmoid(mlp(encode(coords))).reshape(H, W, 3).cpu()
fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))
ax[0].imshow(target.cpu()); ax[0].set_title("target"); ax[0].axis("off")
ax[1].imshow(img.clamp(0, 1)); ax[1].set_title("hash-grid fit"); ax[1].axis("off")
ax[2].plot(*zip(*hist), "-o"); ax[2].set_title("PSNR ↑"); ax[2].set_xlabel("step"); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/B_hashgrid_instngp/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/B_hashgrid_instngp"; os.makedirs(run, exist_ok=True)
torch.save({"tables": tables.detach().cpu(), "mlp": mlp.state_dict()}, f"{run}/hashgrid.pt")
json.dump({"psnr": hist}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/ropedia-" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
Speeds up the NeRF / scene models used in **D · Scene / world**.

### Where to go next
- Feed this encoding into a NeRF (3D coords) → **Instant-NGP** speed for the from-scratch NeRF lab.
- Add per-level learning-rate / weight decay; the real implementation uses a CUDA kernel (tiny-cuda-nn).